# African Wildlife object detection with YOLO11n

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sawayaka143/DL-bonus-project/blob/main/notebooks/train_and_evaluate.ipynb)

This notebook runs the whole project from a fresh Colab runtime:
1. install pinned dependencies
2. download the dataset, remove cross-split duplicates, check the labels
3. train the baseline (imgsz 640) and the experiment (imgsz 320)
4. compare both runs on the validation set, evaluate the selected run on the test set
5. show predictions and failure cases

**Runtime → Change runtime type → T4 GPU** before running.
Training both runs takes roughly 45 min on a T4. Set `TRAIN = False` to skip training and use the
checkpoints we trained (downloaded from the GitHub release) instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('DL-bonus-project'):
    !git clone https://github.com/sawayaka143/DL-bonus-project.git
%cd DL-bonus-project
!pip install -q -r requirements.txt
import torch, ultralytics
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available())

## 1. Data: download, de-duplicate the split, check labels

In [ ]:
!python scripts/download_data.py
!python scripts/make_clean_split.py
!python scripts/data_check.py --data african-wildlife --out results/data/original > /dev/null
!python scripts/data_check.py

In [ ]:
import json
from IPython.display import Image, display, Markdown
stats = json.load(open('results/data/clean/stats.json'))
rows = ['| split | images | objects | buffalo | elephant | rhino | zebra |', '|---|---|---|---|---|---|---|']
for s in ('train', 'val', 'test'):
    o = stats[s]['objects_per_class']
    rows.append(f"| {s} | {stats[s]['images']} | {stats[s]['objects']} | {o['buffalo']} | {o['elephant']} | {o['rhino']} | {o['zebra']} |")
display(Markdown('
'.join(rows)))
print(open('results/data/split_changes.txt').read()[:1500])
print('Label issues:', open('results/data/clean/label_issues.txt').read())
display(Image('results/data/clean/gt_examples.jpg'))

## 2. Training: baseline (imgsz 640) and controlled experiment (imgsz 320)

All other settings (data split, `yolo11n.pt` initial weights, 50 epochs, batch 16, SGD lr0 0.01, seed 0)
are fixed in `scripts/train.py`.

In [ ]:
TRAIN = True  # False: skip training and download our trained checkpoints instead

if TRAIN:
    !python scripts/train.py --name baseline_640 --imgsz 640
    !python scripts/train.py --name exp_320 --imgsz 320
else:
    BASE = 'https://github.com/sawayaka143/DL-bonus-project/releases/download/v1.0'
    for run in ('baseline_640', 'exp_320'):
        os.makedirs(f'runs/{run}/weights', exist_ok=True)
        !wget -q -O runs/{run}/weights/best.pt {BASE}/{run}_best.pt
!ls -la runs/*/weights

In [ ]:
for run in ('baseline_640', 'exp_320'):
    print(run, open(f'runs/{run}/summary.json').read())
    display(Image(f'runs/{run}/results.png', width=900))

## 3. Validation comparison → model selection → test evaluation

In [ ]:
!python scripts/evaluate.py
display(Markdown(open('results/comparison.md').read()))
t = json.load(open('results/test_metrics.json'))
print(f"TEST ({t['run']}): precision {t['precision']:.3f}  recall {t['recall']:.3f}  mAP@0.50 {t['mAP50']:.3f}  mAP@0.50:0.95 {t['mAP50-95']:.3f}")
print('per class:', json.dumps(t['per_class'], indent=1))
display(Image('results/test_confusion_matrix.png', width=600))

## 4. Test predictions and failure cases

In [ ]:
display(Image('results/predictions/test_predictions_grid.jpg'))
f = json.load(open('results/failures/failures.json'))
print(json.dumps(f['summary'], indent=1))
import glob
for p in sorted(glob.glob('results/failures/failure_*.jpg')):
    display(Image(p, width=1000))

## 5. Try the final model on your own image

In [ ]:
from ultralytics import YOLO
import glob
sel = json.load(open('results/comparison.json'))['selected']
model = YOLO(f'runs/{sel}/weights/best.pt')
# Replace SOURCE with any image path or URL (e.g. upload a file in the Colab file browser).
SOURCE = sorted(glob.glob('datasets/african-wildlife-clean/images/test/*.jpg'))[0]
res = model.predict(SOURCE, imgsz=int(sel.split('_')[-1]), conf=0.25)
res[0].save('own_image_pred.jpg')
display(Image('own_image_pred.jpg'))